# 00 — Smoke Test

Run every cell. Each should print `✅ OK`. If anything fails, see the troubleshooting section in the README.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# When jupyter is launched from the repo root, the cwd here is notebooks/.
# Add the repo root so we can import `baml_client` (generated at repo root).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")
key = os.getenv("OPENAI_API_KEY")
assert key and key.startswith("sk-"), "Set OPENAI_API_KEY in .env"
print("✅ OK: OPENAI_API_KEY present")

In [ ]:
from nanny_workshop.openai_client import CachedOpenAI

client = CachedOpenAI(cache_dir=ROOT / ".cache" / "smoke")
reply = client.complete(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Reply with the single word: pong"}],
)
assert "pong" in reply.lower()
print(f"✅ OK: OpenAI completion ({reply!r})")

In [ ]:
vec = client.embed(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    text="nanny",
)
assert len(vec) > 100
print(f"✅ OK: embedding length = {len(vec)}")

In [ ]:
import tempfile
from nanny_workshop.chroma_client import NannyChroma

with tempfile.TemporaryDirectory() as tmp:
    db = NannyChroma(persist_dir=tmp, collection_name="smoke")
    db.add(ids=["a"], documents=["x"], metadatas=[{"k": "v"}], embeddings=[vec])
    out = db.query(query_embedding=vec, n_results=1)
    assert out["ids"][0][0] == "a"
print("✅ OK: Chroma round-trip")

In [ ]:
from baml_client.sync_client import b

result = b.EchoNannyName("test")
assert result.strip().lower() == "test"
print(f"✅ OK: BAML client ({result!r})")

In [ ]:
try:
    from nanny_workshop.phoenix_setup import start_phoenix
    url, _stop = start_phoenix()
    print(f"✅ OK: Phoenix UI at {url}")
except Exception as e:
    print(f"⚠️  Phoenix could not start: {e}. Workshop will fall back to JSON trace logger.")

In [ ]:
pdf_dir = ROOT / "data" / "pdfs"
nannies = sorted(pdf_dir.glob("nanny_resume_*.pdf"))
parents = sorted(pdf_dir.glob("parent_intake_*.pdf"))
assert len(nannies) == 5 and len(parents) == 5, f"expected 5+5 PDFs, got {len(nannies)}+{len(parents)}"
print(f"✅ OK: {len(nannies)} nanny resumes, {len(parents)} parent intakes")

In [ ]:
import json
from nanny_workshop.models import NannyProfile, ParentIntake
data = json.loads((ROOT / "data" / "seed_db.json").read_text())
assert len(data["nannies"]) == 5 and len(data["parents"]) == 5
for n in data["nannies"]: NannyProfile.model_validate(n)
for p in data["parents"]: ParentIntake.model_validate(p)
print("✅ OK: seed_db.json valid (5 nannies, 5 parents)")

## All green?

If every cell above printed ✅ (or Phoenix printed ⚠️ but you don't mind running without observability), you're ready for the workshop.